# Decode-Length Prediction

**Prerequisites:** Python basics and the relevant model-serving concepts.  **Estimated time:** 35 minutes.

## Objectives

- Calibrate length predictions and quantify underprediction and tail reservation.
- Explain at least one failure boundary and a diagnostic.
- Run deterministic assertions against a small synthetic example.


## Model and assumptions

A fixed trace contrasts average error with a high-quantile capacity target. All quantities below are deliberately small, deterministic teaching values. The paired local implementations are dependency-light (the MoE example uses NumPy) and do not call a model or network service. A synthetic proxy establishes accounting behavior, not production quality or speed.


In [ ]:
# Load the paired standalone implementation from the repository root.
import runpy
impl = runpy.run_path('modern-ai/implementations/35-decode-length-prediction.py')


## Transparent baseline

Inspect the short implementation above: the critical state and accounting are visible, so you can change one assumption at a time. In a real system, replace the toy inputs with representative traces and validate quality as well as resource use.


In [ ]:
pred=[7,10,9,11,15,15,18,22,27,30]; actual=[8,9,10,12,13,18,20,44,51,63]
cal=impl['isotonic_non_decreasing'](pred,actual); report=impl['tail_report'](pred,actual)
print('calibrated sorted-bin means',cal,'tail report',report)
assert report['underprediction_rate'] > 0 and report['reserve_p95'] >= max(actual)


## Comparison and metric

Use the printed counts, mask density, error, latency distribution, or decision labels to compare the alternatives. The selected metric is a proxy for a specific resource or behavior; it is not a substitute for end-to-end evaluation.

In [ ]:
# Edge case: reject invalid inputs or expose degraded behavior instead of hiding it.
try:
    impl['tail_report']([1,2],[1])
except (ValueError, MemoryError, IndexError) as exc:
    print(type(exc).__name__, exc)


## Failure cases and exercise

1. Change the topic parameter that exercises a boundary. Predict which metric changes before running the cell.
2. Identify a case where the proxy gives a misleading decision. State what real trace or quality label you would collect before deployment.

**Exercise:** Use the invariant below as a starting test, then add a second case that probes a boundary relevant to your workload.

In [ ]:
assert impl['isotonic_non_decreasing']([1,2,3],[2,1,4]) == [1.5,1.5,4.0]


## Summary and next lessons

This notebook made one decode-length prediction mechanism inspectable with deterministic accounting. Revisit the concept note at `../concepts/35-decode-length-prediction.md`, then compare with the neighboring serving lessons and evaluate on a representative workload before making a system choice.
